## Feature Engineering — Pipelines de preparación de datos

### By:
Miguel Angel Toro Ramos

### Date:
2026-08-21

### Description:

Construcción del proceso de limpieza, transformación y codificación de los datos de admisiones
mediante transformadores y pipelines de scikit-learn. El resultado es un `ColumnTransformer`
reutilizable que produce la matriz de entrada para el entrenamiento de un modelo de regresión
sobre `Chance of Admit `.

## 📚 Import  libraries

In [1]:
# base libraries for data science
from pathlib import Path

import numpy as np
import pandas as pd
import sklearn as sk
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OrdinalEncoder, StandardScaler

## 💾 Load data

In [2]:
# resolve the project root by walking up until pyproject.toml is found,
# so the notebook works regardless of the directory Jupyter was launched from
PROJECT_DIR = next(
    p
    for p in [Path.cwd().resolve(), *Path.cwd().resolve().parents]
    if (p / "pyproject.toml").exists()
)
DATA_DIR = PROJECT_DIR / "data"

admisiones_df = pd.read_parquet(
    DATA_DIR / "02_intermediate/admisiones_type_fixed.parquet", engine="pyarrow"
)

In [3]:
# print library version for reproducibility

print("Pandas version: ", pd.__version__)
print("sklearn version: ", sk.__version__)

Pandas version:  3.0.5
sklearn version:  1.9.0


## 👷 Data preparation or Feature Engineering

### Missing values

Conteo de valores faltantes por columna antes de cualquier transformación.

In [4]:
admision_feature = admisiones_df.copy()

In [5]:
admision_feature.isna().sum()

GRE Score            11
TOEFL Score          19
University Rating     6
SOP                  16
LOR                   8
CGPA                  2
Research             29
Chance of Admit       0
dtype: int64

### Duplicate data

In [6]:
duplicate_rows = admision_feature.duplicated().sum()
print("Number of duplicate rows: ", duplicate_rows)

Number of duplicate rows:  152


In [7]:
admision_feature = admision_feature.drop_duplicates()

In [8]:
admision_feature.info()

<class 'pandas.DataFrame'>
Index: 471 entries, 0 to 581
Data columns (total 8 columns):
 #   Column             Non-Null Count  Dtype  
---  ------             --------------  -----  
 0   GRE Score          460 non-null    Int64  
 1   TOEFL Score        452 non-null    Int64  
 2   University Rating  465 non-null    Int64  
 3   SOP                455 non-null    float64
 4   LOR                463 non-null    float64
 5   CGPA               469 non-null    float64
 6   Research           442 non-null    boolean
 7   Chance of Admit    471 non-null    float64
dtypes: Int64(3), boolean(1), float64(4)
memory usage: 31.7 KB


### Duplicados ocultos tras valores faltantes

`drop_duplicates()` compara valor a valor, y `NaN` nunca es igual a `NaN`. Dos registros idénticos
en los que uno perdió un dato no se detectan como duplicados, aunque lo sean: la comparación falla
justamente en la columna incompleta.

El criterio que sí los detecta es la **compatibilidad en el solapamiento observado**: dos filas son
el mismo registro cuando coinciden en todas las columnas donde ambas tienen dato. No calcula ningún
estadístico del conjunto —solo compara filas entre sí— por lo que puede aplicarse antes de
particionar sin introducir fuga de información.

In [9]:
def marcar_duplicados_compatibles(frame):
    """Flag rows that repeat an earlier record.

    Two rows are treated as the same record when they agree on every column where
    both hold an observed value. Rows differing only in the position of a missing
    value are therefore detected, unlike `DataFrame.duplicated()`, for which NaN is
    never equal to NaN. The first occurrence of each group is kept.
    """
    values = frame.astype({column: "float64" for column in frame.columns}).to_numpy()
    observed = ~np.isnan(values)
    is_duplicate = np.zeros(len(values), dtype=bool)

    for i in range(len(values)):
        if is_duplicate[i]:
            continue
        both = observed[i] & observed[i + 1 :]
        same = np.where(both, values[i] == values[i + 1 :], True).all(axis=1) & both.any(axis=1)
        is_duplicate[np.nonzero(same)[0] + i + 1] = True

    return pd.Series(is_duplicate, index=frame.index)

In [10]:
duplicados_ocultos = marcar_duplicados_compatibles(admision_feature)
print("Filas antes:", len(admision_feature))
print("Duplicados ocultos tras un valor faltante:", int(duplicados_ocultos.sum()))

Filas antes: 471
Duplicados ocultos tras un valor faltante: 71


Un ejemplo del patrón detectado: dos filas idénticas en las que la segunda perdió un valor.

In [11]:
def buscar_grupo(frame, etiqueta):
    """Return the labels of every row compatible with the given one."""
    valores = frame.astype({column: "float64" for column in frame.columns})
    candidata = valores.loc[etiqueta].to_numpy()
    matriz = valores.to_numpy()
    ambos = ~np.isnan(matriz) & ~np.isnan(candidata)
    igual = np.where(ambos, matriz == candidata, True).all(axis=1) & ambos.any(axis=1)
    return frame.index[igual]


primera_duplicada = admision_feature.index[duplicados_ocultos][0]
admision_feature.loc[buscar_grupo(admision_feature, primera_duplicada)]

,GRE Score,TOEFL Score,University Rating,SOP,LOR,CGPA,Research,Chance of Admit
75,329,114,2,2.0,4.0,8.56,True,0.72
403,329,114,2,2.0,NaN,8.56,True,0.72


In [12]:
admision_feature = admision_feature[~duplicados_ocultos]
admision_feature.info()

<class 'pandas.DataFrame'>
Index: 400 entries, 0 to 399
Data columns (total 8 columns):
 #   Column             Non-Null Count  Dtype  
---  ------             --------------  -----  
 0   GRE Score          400 non-null    Int64  
 1   TOEFL Score        400 non-null    Int64  
 2   University Rating  400 non-null    Int64  
 3   SOP                400 non-null    float64
 4   LOR                400 non-null    float64
 5   CGPA               400 non-null    float64
 6   Research           400 non-null    boolean
 7   Chance of Admit    400 non-null    float64
dtypes: Int64(3), boolean(1), float64(4)
memory usage: 27.0 KB


El conjunto queda en 400 registros distintos y **sin valores faltantes**: los 91 nulos del archivo
original estaban íntegramente en las copias descartadas. El tamaño resultante coincide con el del
conjunto público *Graduate Admissions*, del que procede este archivo, lo que confirma que las 223
filas eliminadas —152 duplicados exactos y 71 ocultos tras un faltante— son copias introducidas
durante la preparación del material, no observaciones reales.

Los imputadores se conservan en el pipeline aunque este conjunto ya no los necesite: forman parte
del contrato de preparación y deben seguir operativos frente a datos nuevos que sí presenten
ausencias.

In [13]:
print("Valores faltantes tras la deduplicación:", int(admision_feature.isna().sum().sum()))

Valores faltantes tras la deduplicación: 0


### 👨‍🏭 Feature Engineering

Las columnas se agrupan según el tratamiento que requieren:

- **Numéricas continuas y ordinales**: imputación por mediana y estandarización. `University Rating`,
  `SOP` y `LOR ` son escalas ordinales cuyo orden es informativo, por lo que se tratan como
  numéricas en lugar de codificarse como categorías nominales.
- **Binaria**: `Research` se codifica con `OrdinalEncoder` a valores 0/1 y se imputa por moda.

Cada grupo se resuelve con un `Pipeline` propio y ambos se combinan en un `ColumnTransformer`, de
modo que la preparación completa queda en un único objeto reutilizable.

In [14]:
cols_numeric = ["GRE Score", "TOEFL Score", "University Rating", "SOP", "LOR ", "CGPA"]
cols_binary = ["Research"]

In [15]:
numeric_pipe = Pipeline(
    steps=[
        ("imputer", SimpleImputer(strategy="median")),
        ("scaler", StandardScaler()),
    ]
)

binary_pipe = Pipeline(
    steps=[
        ("encoder", OrdinalEncoder()),
        ("imputer", SimpleImputer(strategy="most_frequent")),
    ]
)

preprocessor = ColumnTransformer(
    transformers=[
        ("numeric", numeric_pipe, cols_numeric),
        ("binary", binary_pipe, cols_binary),
    ]
)

In [16]:
preprocessor

,"transformers transformers: list of tuplesList of (name, transformer, columns) tuples specifying thetransformer objects to be applied to subsets of the data.name : str Like in Pipeline and FeatureUnion, this allows the transformer and its parameters to be set using ``set_params`` and searched in grid search.transformer : {'drop', 'passthrough'} or estimator Estimator must support :term:`fit` and :term:`transform`. Special-cased strings 'drop' and 'passthrough' are accepted as well, to indicate to drop the columns or to pass them through untransformed, respectively.columns : str, array-like of str, int, array-like of int, array-like of bool, slice or callable Indexes the data on its second axis. Integers are interpreted as positional columns, while strings can reference DataFrame columns by name. A scalar string or int should be used where ``transformer`` expects X to be a 1d array-like (vector), otherwise a 2d array will be passed to the transformer. A callable is passed the input data `X` and can return any of the above. To select multiple columns by name or dtype, you can use :obj:`make_column_selector`.","[('numeric', ...), ('binary', ...)]"
,"remainder remainder: {'drop', 'passthrough'} or estimator, default='drop'By default, only the specified columns in `transformers` aretransformed and combined in the output, and the non-specifiedcolumns are dropped. (default of ``'drop'``).By specifying ``remainder='passthrough'``, all remaining columns thatwere not specified in `transformers`, but present in the data passedto `fit` will be automatically passed through. This subset of columnsis concatenated with the output of the transformers. For dataframes,extra columns not seen during `fit` will be excluded from the outputof `transform`.By setting ``remainder`` to be an estimator, the remainingnon-specified columns will use the ``remainder`` estimator. Theestimator must support :term:`fit` and :term:`transform`.Note that using this feature requires that the DataFrame columnsinput at :term:`fit` and :term:`transform` have identical order.",'drop'
,"sparse_threshold sparse_threshold: float, default=0.3If the output of the different transformers contains sparse matrices,these will be stacked as a sparse matrix if the overall density islower than this value. Use ``sparse_threshold=0`` to always returndense. When the transformed output consists of all dense data, thestacked result will be dense, and this keyword will be ignored.",0.3
,"n_jobs n_jobs: int, default=NoneNumber of jobs to run in parallel.``None`` means 1 unless in a :obj:`joblib.parallel_backend` context.``-1`` means using all processors. See :term:`Glossary <n_jobs>`for more details.",None
,"transformer_weights transformer_weights: dict, default=NoneMultiplicative weights for features per transformer. The output of thetransformer is multiplied by these weights. Keys are transformer names,values the weights.",None
,"verbose verbose: bool, default=FalseIf True, the time elapsed while fitting each transformer will beprinted as it is completed.",False
,"verbose_feature_names_out verbose_feature_names_out: bool, str or Callable[[str, str], str], default=True- If True, :meth:`ColumnTransformer.get_feature_names_out` will prefix all feature names with the name of the transformer that generated that feature. It is equivalent to setting `verbose_feature_names_out=""{transformer_name}__{feature_name}""`.- If False, :meth:`ColumnTransformer.get_feature_names_out` will not prefix any feature names and will error if feature names are not unique.- If ``Callable[[str, str], str]``, :meth:`ColumnTransformer.get_feature_names_out` will rename all the features using the name of the transformer. The first argument of the callable is the transformer name and the second argument is the feature name. The returned string will be the new feature name.- If ``str``, it must be a string ready for formatting. The given string will be formatted using two field names: ``transformer_name`` and ``feature

### Train / Test split

In [17]:
X_features = admision_feature.drop(columns=["Chance of Admit "])
Y_target = admision_feature["Chance of Admit "]

# 80% train, 20% test
y_bins = pd.qcut(Y_target, q=5, labels=False, duplicates="drop")
x_train, x_test, y_train, y_test = train_test_split(
    X_features, Y_target, test_size=0.2, random_state=42, stratify=y_bins
)

In [18]:
x_train.shape, y_train.shape

((320, 7), (320,))

In [19]:
x_test.shape, y_test.shape

((80, 7), (80,))

### Preprocessing pipeline

In [20]:
# fit the preprocessor on the training set only, to avoid data leakage
preprocessor.fit(x_train)

,"transformers transformers: list of tuplesList of (name, transformer, columns) tuples specifying thetransformer objects to be applied to subsets of the data.name : str Like in Pipeline and FeatureUnion, this allows the transformer and its parameters to be set using ``set_params`` and searched in grid search.transformer : {'drop', 'passthrough'} or estimator Estimator must support :term:`fit` and :term:`transform`. Special-cased strings 'drop' and 'passthrough' are accepted as well, to indicate to drop the columns or to pass them through untransformed, respectively.columns : str, array-like of str, int, array-like of int, array-like of bool, slice or callable Indexes the data on its second axis. Integers are interpreted as positional columns, while strings can reference DataFrame columns by name. A scalar string or int should be used where ``transformer`` expects X to be a 1d array-like (vector), otherwise a 2d array will be passed to the transformer. A callable is passed the input data `X` and can return any of the above. To select multiple columns by name or dtype, you can use :obj:`make_column_selector`.","[('numeric', ...), ('binary', ...)]"
,"remainder remainder: {'drop', 'passthrough'} or estimator, default='drop'By default, only the specified columns in `transformers` aretransformed and combined in the output, and the non-specifiedcolumns are dropped. (default of ``'drop'``).By specifying ``remainder='passthrough'``, all remaining columns thatwere not specified in `transformers`, but present in the data passedto `fit` will be automatically passed through. This subset of columnsis concatenated with the output of the transformers. For dataframes,extra columns not seen during `fit` will be excluded from the outputof `transform`.By setting ``remainder`` to be an estimator, the remainingnon-specified columns will use the ``remainder`` estimator. Theestimator must support :term:`fit` and :term:`transform`.Note that using this feature requires that the DataFrame columnsinput at :term:`fit` and :term:`transform` have identical order.",'drop'
,"sparse_threshold sparse_threshold: float, default=0.3If the output of the different transformers contains sparse matrices,these will be stacked as a sparse matrix if the overall density islower than this value. Use ``sparse_threshold=0`` to always returndense. When the transformed output consists of all dense data, thestacked result will be dense, and this keyword will be ignored.",0.3
,"n_jobs n_jobs: int, default=NoneNumber of jobs to run in parallel.``None`` means 1 unless in a :obj:`joblib.parallel_backend` context.``-1`` means using all processors. See :term:`Glossary <n_jobs>`for more details.",None
,"transformer_weights transformer_weights: dict, default=NoneMultiplicative weights for features per transformer. The output of thetransformer is multiplied by these weights. Keys are transformer names,values the weights.",None
,"verbose verbose: bool, default=FalseIf True, the time elapsed while fitting each transformer will beprinted as it is completed.",False
,"verbose_feature_names_out verbose_feature_names_out: bool, str or Callable[[str, str], str], default=True- If True, :meth:`ColumnTransformer.get_feature_names_out` will prefix all feature names with the name of the transformer that generated that feature. It is equivalent to setting `verbose_feature_names_out=""{transformer_name}__{feature_name}""`.- If False, :meth:`ColumnTransformer.get_feature_names_out` will not prefix any feature names and will error if feature names are not unique.- If ``Callable[[str, str], str]``, :meth:`ColumnTransformer.get_feature_names_out` will rename all the features using the name of the transformer. The first argument of the callable is the transformer name and the second argument is the feature name. The returned string will be the new feature name.- If ``str``, it must be a string ready for formatting. The given string will be formatted using two field names: ``transformer_name`` and ``feature

In [21]:
feature_names = preprocessor.get_feature_names_out()

x_train_transformed = pd.DataFrame(
    preprocessor.transform(x_train), columns=feature_names, index=x_train.index
)
x_test_transformed = pd.DataFrame(
    preprocessor.transform(x_test), columns=feature_names, index=x_test.index
)
x_train_transformed.info()

<class 'pandas.DataFrame'>
Index: 320 entries, 190 to 314
Data columns (total 7 columns):
 #   Column                      Non-Null Count  Dtype  
---  ------                      --------------  -----  
 0   numeric__GRE Score          320 non-null    float64
 1   numeric__TOEFL Score        320 non-null    float64
 2   numeric__University Rating  320 non-null    float64
 3   numeric__SOP                320 non-null    float64
 4   numeric__LOR                320 non-null    float64
 5   numeric__CGPA               320 non-null    float64
 6   binary__Research            320 non-null    float64
dtypes: float64(7)
memory usage: 20.0 KB


In [22]:
x_train_transformed.head()

,numeric__GRE Score,numeric__TOEFL Score,numeric__University Rating,numeric__SOP,numeric__LOR,numeric__CGPA,binary__Research
190,0.647835,0.612718,1.675047,1.071412,0.628869,0.945473,1.0
7,-0.736520,-1.029408,-0.943919,-0.400055,0.628869,-1.170741,0.0
227,-0.390431,0.448506,-0.943919,0.090434,-0.479881,-0.112634,0.0
393,0.042180,-0.536770,-0.943919,-0.400055,-0.479881,0.273659,0.0
215,1.166968,1.433782,1.675047,1.561901,1.183243,1.281380,1.0


In [23]:
x_train.head()

,GRE Score,TOEFL Score,University Rating,SOP,LOR,CGPA,Research
190,324,111,5,4.5,4.0,9.16,True
7,308,101,2,3.0,4.0,7.90,False
227,312,110,2,3.5,3.0,8.53,False
393,317,104,2,3.0,3.0,8.76,False
215,330,116,5,5.0,4.5,9.36,True


## 📊 Analysis of Results and Conclusions

El pipeline de preparación se construyó sobre `admisiones_type_fixed.parquet` (623 filas, 8
columnas) y produce una matriz de entrenamiento lista para modelar.

**Limpieza: duplicados exactos.** La eliminación de filas idénticas redujo el conjunto de 623 a 471:
se descartaron 152 registros, el 24.4% del total.

**Limpieza: duplicados ocultos tras valores faltantes.** `drop_duplicates()` no agota el problema.
Compara valor a valor y `NaN` nunca es igual a `NaN`, de modo que dos registros idénticos en los que
uno perdió un dato no se detectan como duplicados: la comparación falla precisamente en la columna
incompleta. Aplicando compatibilidad en el solapamiento observado —dos filas son el mismo registro
cuando coinciden en todas las columnas donde ambas tienen dato— aparecen 71 duplicados adicionales.
El conjunto queda en **400 registros distintos**, frente a las 471 que sugería la deduplicación
exacta.

Este criterio no calcula ningún estadístico del conjunto, solo compara filas entre sí, por lo que se
aplica antes de particionar sin introducir fuga de información.

**Valores faltantes.** Los 91 nulos del archivo original estaban **íntegramente en las copias
descartadas**: tras la deduplicación el conjunto no presenta ninguna ausencia. Ese dato, junto con
que las 400 filas restantes coinciden con el tamaño del conjunto público *Graduate Admissions* del
que procede el archivo, indica que las 223 filas eliminadas son copias introducidas durante la
preparación del material y no observaciones reales.

Los imputadores se conservan en el `ColumnTransformer` pese a que este conjunto ya no los necesite.
Forman parte del contrato de preparación y deben permanecer operativos frente a datos nuevos que sí
presenten ausencias; eliminarlos dejaría el pipeline inservible en producción. En este conjunto son
inertes, y el notebook lo hace explícito en lugar de dar por hecho que actúan.

**Partición.** La división 80/20 con `random_state=42` produce 320 filas de entrenamiento y 80 de
prueba. Como el objetivo es continuo, la estratificación se aplicó sobre quintiles construidos con
`pd.qcut` y no sobre el valor crudo: estratificar directamente sobre los valores únicos de
`Chance of Admit ` falla, porque varios de ellos aparecen una sola vez.

**Prevención de fuga de información.** El `preprocessor` se ajusta exclusivamente con `x_train`. La
media y la desviación del escalador, y los estadísticos de los imputadores, provienen solo del
conjunto de entrenamiento y se aplican sobre prueba sin recalcularse. Ajustar cualquiera de estos
transformadores sobre el conjunto completo habría filtrado información del conjunto de prueba y
producido una estimación optimista del desempeño.

La deduplicación completa cierra además una segunda vía de fuga, menos evidente: mientras
sobrevivían duplicados ocultos, un mismo registro podía quedar repartido entre entrenamiento y
prueba, de modo que parte del conjunto de prueba ya había sido vista durante el ajuste.

**Codificación.** `Research` es booleana y representa una condición binaria. Se codifica con
`OrdinalEncoder`, que la lleva a 0/1 preservando los nulos para que el imputador posterior los
resuelva por moda. Se descartó `OneHotEncoder` porque sobre una variable de dos categorías genera
dos columnas perfectamente colineales, donde una es el complemento de la otra: información
redundante que deja singular la matriz de diseño en modelos lineales. `University Rating`, `SOP` y
`LOR ` son escalas ordinales cuyo orden es informativo, por lo que se conservan como numéricas. El
conjunto no contiene variables categóricas nominales ni de texto, de modo que no se requirió otra
codificación.

**Escalado.** Las variables numéricas se estandarizan con `StandardScaler` porque sus rangos son
heterogéneos: GRE y TOEFL están en el orden de las centenas, CGPA en el de las unidades y `Research`
vale 0 o 1. Sin estandarizar, los algoritmos basados en distancias o en regularización quedan
dominados por la variable de mayor magnitud. La comparación entre `x_train.head()` y
`x_train_transformed.head()` muestra el efecto: las seis columnas numéricas quedan centradas en 0.

**Discretización y transformaciones no lineales.** El requerimiento las contempla "cuando sea
apropiado" y en este conjunto no lo son. El análisis univariable describe relaciones monótonas y
aproximadamente lineales entre los predictores y el objetivo, con asimetría cercana a cero en todos
ellos: no hay efectos de umbral que la discretización pueda capturar ni sesgo que corrijan las
transformaciones logarítmicas o de raíz. Discretizar habría degradado la resolución de los
predictores más informativos sin contrapartida.

**Selección de atributos.** No se aplicó. El paso es opcional en el requerimiento y, con solo seis
predictores, descartar alguno requeriría evidencia de desempeño que corresponde a la etapa de
modelado.

**Resultado.** La matriz final tiene 7 columnas, 320 filas de entrenamiento y 80 de prueba, y no
contiene valores nulos. La preparación completa queda encapsulada en un único objeto
(`preprocessor`), aplicable a datos nuevos con la misma secuencia de transformaciones sin
reproducir los pasos manualmente.

**Limitaciones.** La muestra útil es de 400 registros, un 36% menos que las 623 filas del archivo de
partida. Con 80 observaciones de prueba, cualquier métrica calculada sobre esa partición tendrá
varianza considerable, por lo que la evaluación de modelos debe apoyarse en validación cruzada y no
en una medición única.

## 💡 Proposals and Ideas

- Documentar en la etapa de adquisición que el archivo de partida contiene 223 filas duplicadas
  sobre 400 registros reales, para que futuras cargas no reintroduzcan el problema.
- Trasladar la deduplicación por compatibilidad a la etapa de datos intermedios, de modo que
  `admisiones_type_fixed.parquet` se publique ya sin duplicados y ningún notebook posterior deba
  repetirla.
- Normalizar los nombres de columna: `LOR ` y `Chance of Admit ` tienen espacios finales, lo que
  obliga a replicarlos literalmente en todo el código y es una fuente silenciosa de `KeyError`.
- Comparar `StandardScaler` contra `MinMaxScaler` y `RobustScaler` midiendo el efecto sobre un
  modelo base, en lugar de fijar el escalador por convención.
- Reevaluar la discretización y las transformaciones no lineales con modelos de otra familia
  (árboles, gradient boosting), donde el criterio de linealidad usado aquí no aplica.
- Aplicar selección de atributos y verificar si `SOP` y `LOR ` aportan información más allá de la
  ya contenida en CGPA y GRE.
- Serializar el `preprocessor` ajustado y las particiones transformadas para que la etapa de
  modelado las consuma sin repetir el ajuste.
- Sustituir la partición simple por validación cruzada con `KFold`, dado que 80 filas de prueba
  producen estimaciones de error con varianza considerable.

## 📖 References

- [Ciencia de datos con Python — Feature Engineering, José R. Zapata](https://joserzapata.github.io/post/ciencia-datos-proyecto-python/4-feat_eng/)
- [scikit-learn — Preprocessing data](https://scikit-learn.org/stable/modules/preprocessing.html)
- [scikit-learn — Pipelines and composite estimators](https://scikit-learn.org/stable/modules/compose.html)
- [scikit-learn — Common pitfalls: data leakage](https://scikit-learn.org/stable/common_pitfalls.html#data-leakage)